# Basics of IRT
Source: [EACL Tutorial (2024)](https://eacl2024irt.github.io/)

## What do we evaluate in NLP? 

Evaluations can be at several levls: 

1. System level Evaluations (Eg: MT System, Chatbot etc)
2. Method Evaluations (Eg: Which model architecture is better LSTM vs Transformer?)
3. Metric Evaluations (Eg: Which metric is consistent with our task BLEU, BERTScore, etc?)
4. Annotation Evaluations (Eg: What errors do they have? What is the quality of annotations?)
5. Data Evaluations (Eg: How domain relevant is data?)

Different systems can have different requirements but we have some common task frameworks (CTF), like benchmarks, shared tasks & leaderboards, which measure general progress of systems. 

- [Schwitter et al (2000)](https://dl.acm.org/doi/10.3115/1117595.1117599): There is general agreement that competitive evals had striking and beneficial effect on performance of various systems. But can lead to cases where participating systems narrowly optimize for these tasks.
- [Lipton and Steinhardt (2019)](https://spawn-queue.acm.org/doi/full/10.1145/3317287.3328534): "Focusing on SOTA numbers provides limited value for scientific progress absent insight what drives them". 

So, how do we improve CTF and make them more insightful? 

- Ask questions which have right amount of difficulty and are discriminative.
- Maximize fairness across all tested models.
- Appropriate model rankings on leaderboards.

What are the common methods of ranking? 

1. Average Score
2. Z-Score Ratings
3. Preference Rankings (Bradley-Terry-Leech, ELO Ratings (AB Testing), Trueskill Ratings, Item Response Theory)

Hence, Item Response Theory (IRT) can help provide a framework for: 

1. Better comparison of systems (identify appropriate questions (quality, difficulty, discrimination), adequate comparison, informed rankings)
2. Better error analysis (error sources, error types, error explanations)

## Why do we need IRT? 

In a given evaluation dataset, not all items are equally informative of model capabilities or for distinguishing models! 

<img src="images/QDiff.png" 
        alt="Picture" 
        width="400" 
        height="400" 
        style="display: block; margin: 0 auto" />


We need a way to pick out discriminative questions (not too easy, not too hard, and not incorrect due to annotation error).


<img src="images/QDisc.png" 
        alt="Picture" 
        width="400" 
        height="400" 
        style="display: block; margin: 0 auto" />

IRT helps us do this!


Source: [EACL Tutorial Video](youtube.com/watch?v=BJ7QumB30iI&sttick=0)

## Defining IRT

IRT has origins in psychometrics i.e the study of quantitative measurement practices of human psychology which: 

1. Builds instruments of measurement (standardized tests)
2. Develops theoretical approaches to measurement

IRT measures latent traits of test-takers with respect to given test questions ("items"). It gives us: 

- **Values of Items**:  It gives us discrimative power or difficulty of an item such that similar value items have similar probability of being answered correctly and can be swapped out. 
- **Scores**: A way to score systems with the most valuable items such that we get actual insight into ability of model.
- **Efficiency**: A way to find most valued items and build smaller evaluation sets.

# Types of IRT Models

## 1 PL Model (Rasch Model) 

It models the probability that a model $j$ answers an item $i$ correctly as a function of two parameters: latent ability ($\theta_j$) and difficulty ($b_i$). 

$$p(y_{ij}=1 \mid b_i, \theta_j)= \frac{1}{1+ e^{-(\theta_j-b_i)}}$$


If we then plot $p(y_{ij})$ vs $\theta_j$, different items will have different curves since they have different difficulties $b_i$. These curves are known as Item Characteristic Curves (ICC). In the figure below, orange represents a lower difficulty item whereas blue is higher difficulty item (for a given $\theta_j$, orange has higher probability of correctness and thus lower x-axis offset))

<img src="images/ICCRasch.png" 
        alt="Picture" 
        width="200" 
        height="200" 
        style="display: block; margin: 0 auto" />



## 2 PL Model

It models the probability that a model $j$ answers an item $i$ correctly as a function of three parameters: latent ability ($\theta_j$), difficulty ($b_i$), and discriminability ($a_i$). 

$$p(y_{ij}=1 \mid a_i, b_i, \theta_j)= \frac{1}{1+ e^{-a_i(\theta_j-b_i)}}$$

If we now plot $p(y_{ij})$ vs $\theta_j$ to get ICC, difficulties $b_i$ emerge as x-axis offset while discriminability $a_i$ emerges as slope of the curve (higher slope, higher discriminability).

<img src="images/ICC2PL.png" 
        alt="Picture" 
        width="200" 
        height="200" 
        style="display: block; margin: 0 auto" />

## 3 PL Model

It models the probability that a model $j$ answers an item $i$ correctly as a function of four parameters: latent ability ($\theta_j$), difficulty ($b_i$), discriminability ($a_i$), and guessing ($c_i$) . 

$$p(y_{ij}=1 \mid a_i, b_i, c_i,  \theta_j)= \frac{1- c_i}{1+ e^{-a_i(\theta_j-b_i)}}$$

If we now plot $p(y_{ij})$ vs $\theta_j$ to get ICC, difficulties $b_i$ emerge as x-axis offset, discriminability $a_i$ emerges as slope, and guessing $a_i$ appears as y-axis offset (can have some probability of answering even with low ability) .

<img src="images/ICC3PL.png" 
        alt="Picture" 
        width="200" 
        height="200" 
        style="display: block; margin: 0 auto" />

## Feasibility Model

It models the probability that a model $j$ answers an item $i$ correctly as a function of four parameters: latent ability ($\theta_j$), difficulty ($b_i$), discriminability ($a_i$), and feasibility ($\gamma_i$) . This model is useful for annotation errors where the feasibility of a correct answer is never high due to errors. 

$$p(y_{ij}=1 \mid a_i, b_i, \gamma_i,\theta_j)= \frac{\gamma_i}{1+ e^{-a_i(\theta_j-b_i)}}$$

If we now plot $p(y_{ij})$ vs $\theta_j$ to get ICC, difficulties $b_i$ emerge as x-axis offset, discriminability $a_i$ emerges as slope, and feasibility $\gamma_i$ appears as inverse of guessing (can have low probability of answering even with higher ability) .

<img src="images/ICC4PL.png" 
        alt="Picture" 
        width="200" 
        height="200" 
        style="display: block; margin: 0 auto" />

# Training of IRT Models

## Intuition

Let us consider what we have first: 

1. **Variables**: We have observed binary response variables ($y_{ij}={0,1}$) and unobserved latent variables $b_i$ and $\theta_j$.
2. **Generative Story**: Every person has some latent ability. Every question has some latent difficulty. When a person tries a question, the probability of success depends on how ability compares to difficulty.
3. **Bayesian Story**: We start with some prior beliefs about latent variable values, and update them as we see data to arrive at the posterior belief i.e given all responses what do I believe about all difficulties and all abilities? However, this is intractable.
4. **Variational Inference Guide**: We use a simpler guide model of belief to approximate a distribution, see difference from original distribution and update.
   
**Training Story**: 

- The guide proposes some hidden abilities and difficulties. You check how well those hidden values would explain the observed answers.

- If they explain the data well, you make the guide more confident in them. If they explain the data poorly, you shift the guide.

- Over many iterations, the guide reshapes itself so that: It generates hidden values that explain the data well, It doesn’t become overconfident when data is weak.


**Output**: 

- For each test-taker: a belief about their ability.

- For each item: a belief about its difficulty.

- Uncertainty about both.

## Formalization

**Variables**

Observed variables:
- Responses: $y_{ij} \in \{0,1\} $

Latent variables:
- Item difficulty: $b_i \in \mathbb{R}$
- Person ability: $\theta_j \in \mathbb{R}$

---

**Generative Model**

We assume that responses are generated as:

$$p(y_{ij} = 1 \mid \theta_j, b_i) = \sigma(\theta_j - b_i),
\quad \sigma(x) = \frac{1}{1 + e^{-x}}.$$

Equivalently,

$$p(y_{ij} \mid \theta_j, b_i)
= \sigma(\theta_j - b_i)^{y_{ij}} \left(1 - \sigma(\theta_j - b_i)\right)^{1 - y_{ij}}.$$

---

**Priors and Posterior**

We place priors on the latent variables:

$$\theta_j \sim \mathcal{N}(0,1), \qquad b_i \sim \mathcal{N}(0,1).$$

The joint distribution is:

$$p(y, \theta, b)
= \prod_j p(\theta_j) \prod_i p(b_i) \prod_{i,j} p(y_{ij} \mid \theta_j, b_i).$$

The posterior of interest is:

$$p(\theta, b \mid y)
= \frac{p(y \mid \theta, b) p(\theta) p(b)}{p(y)},$$

where $p(y) = \int p(y \mid \theta, b) p(\theta) p(b)\, d\theta\, db$ is intractable.

---

**Variational Guide**

We approximate the posterior with a factorized variational family:

$$
q(\theta, b)
= \prod_j q(\theta_j) \prod_i q(b_i),
$$

with

$$
q(\theta_j) = \mathcal{N}(\mu_{\theta_j}, \sigma_{\theta_j}^2), \qquad
q(b_i) = \mathcal{N}(\mu_{b_i}, \sigma_{b_i}^2).
$$

The variational parameters $\{\mu_{\theta_j}, \sigma_{\theta_j}, \mu_{b_i}, \sigma_{b_i}\} $ are learned.

---

**Training Objective (ELBO)**

We choose \(q\) to maximize the Evidence Lower Bound (ELBO):

$$\mathcal{L}= \mathbb{E}_{q}[\log p(y \mid \theta, b)]- \sum_j \mathrm{KL}(q(\theta_j) \| p(\theta_j))- \sum_i \mathrm{KL}(q(b_i) \| p(b_i)).$$

---
**Training Procedure**

Repeat until convergence:

1. Sample latent variables from the guide:

   $$
   \theta_j = \mu_{\theta_j} + \sigma_{\theta_j} \epsilon_j,\quad \epsilon_j \sim \mathcal{N}(0,1),
   $$

   $$
   b_i = \mu_{b_i} + \sigma_{b_i} \eta_i,\quad \eta_i \sim \mathcal{N}(0,1).
   $$

2. Compute predicted probabilities:

   $$
   \hat{p}_{ij} = \sigma(\theta_j - b_i).
   $$

3. Evaluate the ELBO and take gradient ascent steps on the variational parameters.

---

**Output**

- Ability Means:  $\boldsymbol{\mu}_\theta = [\mu_{\theta_1}, \mu_{\theta_2}, \dots, \mu_{\theta_N}]$

- Ability Uncertainties: $\boldsymbol{\sigma}_\theta = [\sigma_{\theta_1}, \sigma_{\theta_2}, \dots, \sigma_{\theta_N}]$

- Difficulty Means: $\boldsymbol{\mu}_b = [\mu_{b_1}, \mu_{b_2}, \dots, \mu_{b_M}]$

- Difficulty Uncertainties: $\boldsymbol{\sigma}_b = [\sigma_{b_1}, \sigma_{b_2}, \dots, \sigma_{b_M}]$



How We Use Them

1. $\boldsymbol{\mu}_\theta$ — report and compare person abilities
2. $\boldsymbol{\sigma}_\theta$ — quantify confidence in each ability estimate.  
3. $\boldsymbol{\mu}_b$ — analyze and select item difficulties.  
4. $\boldsymbol{\sigma}_b$ — detect poorly estimated or unstable items.

# Implementation of IRT Models

## Data Loading

In [1]:
# Importing libraries
import sys
import numpy as np
from pydantic import BaseModel

In [2]:
# Creating data classes
# Subjects: 10 with random latent ability
class Subject(BaseModel): 
    subject_id:str
    ability:float
# Items: 1000 with random difficulty and validity rate
class Item(BaseModel): 
    item_id: str
    category:str="all"
    difficulty:float
    valid:bool

In [3]:
# Creating data functions
# Subject Creation Function
min_ability= -4
max_ability=4
def create_subject(subject_id:str):
    return Subject(subject_id=subject_id, ability=np.random.uniform(low=min_ability, high=max_ability))
    
# Item Creation Function
min_diff=-4
max_diff=4
validity_rate=0.9
def create_item(item_id:str, category:str): 
    validity=np.random.uniform()
    if validity>validity_rate:
        valid=0 
    else:
        valid=1
    return Item(item_id= item_id, category=category, difficulty=np.random.uniform(low=min_diff, high=max_diff), valid=valid)

In [4]:
# Generating data
# 10 Subjects 
subjects= [create_subject(f'subject_{idx}') for idx in range(0,10)]
items=[create_item(f'item_{idx}','all') for idx in range(0,1000) ]
print(type(subjects))
print(subjects[0])
print(type(items))
print(items[0])

<class 'list'>
subject_id='subject_0' ability=-3.016081578975454
<class 'list'>
item_id='item_0' category='all' difficulty=-0.5736687864906322 valid=True


## Data Formatting

Py-IRT needs the data to be in jsonlines format as follows: 

Valid Data: `{"subject_id":"<subject_id>", "responses":{"<item_id>":<response>}`

Note: `response` is binary variable here and responses does not need to be complete (not all models answer all questions)

In [5]:
# Importing libraries
import random
import py_irt # pyirt library 
from py_irt.io import write_jsonlines #py_irt package with io.py file with jsonlines function
from py_irt.dataset import Dataset #py_irt package with dataset.py file with Dataset class

In [6]:
# Dataset Alteration function
# takes the list of models, list of benchmark questions, location to store new dataset
def write_irt_dataset(subjects:list[Subject], items:list[Item], path:str): 
     # prints number of models and number of items
    print(f'Subjects:{len(subjects)}, Items:{len(items)}')
    
    # artifacts created by the function
    score_by_subject={} # a dictionary where each key is a model and each value is accuracy
    lookup={} # a dictionary where each key is a model and each value is a dictionary of actual responses for each item
    rows=[] # the final response format created from lookup, input to write_jsonlines function

    # loop through each subject to create a dictionary of responses
    for subject in subjects:
        responses={}
        correct=0
        total=0
        # for each subject, loop through each benchmark item
        for item in items:
            # some items are treated as valid, response is sampled from logistic model
            if item.valid:
                responses[item.item_id]= int(1 / (1 + np.exp(-(subject.ability - item.difficulty))) > random.random())
            # some items are treated as noise, response is sampled randomly
            else:
                responses[item.item_id] = int(random.random() > .5)
            # calculate number of correct responses
            correct += responses[item.item_id]
            # calculate number of total responses
            total += 1
        # calculate score for the subject
        score_by_subject[subject.subject_id] = correct / total
        # create the lookup responses dict for the subject
        lookup[subject.subject_id] = responses
        # create the formatted dict item for the subject
        rows.append({"subject_id": subject.subject_id, "responses": responses})

    # write jsonlines file using the rows list
    write_jsonlines(path, rows)

    # return the score for each subject and the lookup dict (optional)
    return score_by_subject, lookup


In [7]:
# writing the dataset
score_by_subject, subject_responses_dict = write_irt_dataset(subjects, items, 'irt_dataset_section_2.jsonlines')
# checking average accuracy for each subject
print(score_by_subject)

Subjects:10, Items:1000
{'subject_0': 0.204, 'subject_1': 0.559, 'subject_2': 0.165, 'subject_3': 0.485, 'subject_4': 0.345, 'subject_5': 0.318, 'subject_6': 0.353, 'subject_7': 0.688, 'subject_8': 0.377, 'subject_9': 0.185}


## Model Training using PyIRT

In [8]:
# Importing libraries
import torch
from py_irt.dataset import Dataset
from py_irt.config import IrtConfig
from py_irt.training import IrtModelTrainer

In [9]:
# Checking device for training
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(device)

mps


In [10]:
# Loading data
dataset = Dataset.from_jsonlines("irt_dataset_section_2.jsonlines")

[14:39:52] amortized: False                                                                          ]8;id=731367;file:///Users/ruchira/miniforge3/envs/benchfilterenv/lib/python3.10/site-packages/py_irt/dataset.py\dataset.py]8;;\:]8;id=325819;file:///Users/ruchira/miniforge3/envs/benchfilterenv/lib/python3.10/site-packages/py_irt/dataset.py#116\116]8;;\

At present, in PyIRT, the one parameter logistic (1PL) model, aka Rasch model, two parameter logistic model (2PL) and four parameter logistic model (4PL) have been implemented. The user can specify whether vague or hierarchical priors are used. The three-parameter logistic model is in the pipeline and will be added when available.

In [11]:
# Setting up training
# config with model type, log rate epochwise and dropout rate
config = IrtConfig(model_type='1pl', log_every=500, dropout=.2)
# initializing trainer object with config and dataset
trainer = IrtModelTrainer(config=config, data_path=None, dataset=dataset)

[14:39:54] Vocab size: None                                                                          ]8;id=12307;file:///Users/ruchira/miniforge3/envs/benchfilterenv/lib/python3.10/site-packages/py_irt/training.py\training.py]8;;\:]8;id=779407;file:///Users/ruchira/miniforge3/envs/benchfilterenv/lib/python3.10/site-packages/py_irt/training.py#88\88]8;;\

In [12]:
# Training IRT model
# set epochs and device
trainer.train(epochs=10000, device='cpu') # only takes cuda or cpu

[14:39:57] args: {'device': 'cpu', 'num_items': 1000, 'num_subjects': 10}                           ]8;id=618495;file:///Users/ruchira/miniforge3/envs/benchfilterenv/lib/python3.10/site-packages/py_irt/training.py\training.py]8;;\:]8;id=860776;file:///Users/ruchira/miniforge3/envs/benchfilterenv/lib/python3.10/site-packages/py_irt/training.py#138\138]8;;\

           Parsed Model Args: {'device': 'cpu', 'num_items': 1000, 'num_subjects': 10, 'priors':    ]8;id=452089;file:///Users/ruchira/miniforge3/envs/benchfilterenv/lib/python3.10/site-packages/py_irt/training.py\training.py]8;;\:]8;id=78326;file:///Users/ruchira/miniforge3/envs/benchfilterenv/lib/python3.10/site-packages/py_irt/training.py#151\151]8;;\
           'vague', 'dropout': 0.2, 'hidden': 100, 'vocab_size': None}                                             

Training Pyro IRT Model for 10000 epochs

Output()

## Fitted Model

In [15]:
# Actual skill, inferred skill, and accuracy of Subjects
for subject, skill, acc in sorted(list(zip(subjects, trainer.last_params['ability'], score_by_subject.values())), key=lambda v: v[0].ability):
    print(subject.subject_id, "Real Skill", subject.ability, "Inferred Skill", skill, "Acc", acc)

subject_2 Real Skill -3.712206617115135 Inferred Skill -2.4701480865478516 Acc 0.165
subject_9 Real Skill -3.21196145403555 Inferred Skill -2.1192519664764404 Acc 0.185
subject_0 Real Skill -3.016081578975454 Inferred Skill -1.9426631927490234 Acc 0.204
subject_6 Real Skill -1.6664674371156227 Inferred Skill -0.5634949207305908 Acc 0.353
subject_5 Real Skill -1.6566859736429231 Inferred Skill -0.8417225480079651 Acc 0.318
subject_4 Real Skill -1.5021358017460926 Inferred Skill -0.6272015571594238 Acc 0.345
subject_8 Real Skill -1.1655270587055604 Inferred Skill -0.3364512026309967 Acc 0.377
subject_3 Real Skill -0.4026245539390896 Inferred Skill 0.6427916884422302 Acc 0.485
subject_1 Real Skill 0.3373340645151899 Inferred Skill 1.3904953002929688 Acc 0.559
subject_7 Real Skill 1.7633780279291074 Inferred Skill 2.7245404720306396 Acc 0.688


In [16]:
# Difficulty of items
for item, diff in sorted(list(zip(items, trainer.last_params["diff"]))[:20], key=lambda v: v[0].difficulty):
    print(item.difficulty, diff)

-3.9674365718460374 -2.5661985874176025
-3.8585986898942517 -1.853598952293396
-3.752635161265119 -1.7666996717453003
-3.324342962283506 -1.769580602645874
-2.6114925647665634 -1.2843297719955444
-2.168901695889325 -0.9050300717353821
-1.815048501876813 -0.47096994519233704
-0.652425846560674 0.2056436687707901
-0.5736687864906322 0.06972132623195648
-0.4646367892084262 -1.0629717111587524
0.7549375193114338 1.8224904537200928
0.7852491101945747 1.7418569326400757
1.0596154121180943 3.084428071975708
1.0768492869114308 0.904133677482605
1.4569542165707405 2.8970232009887695
1.9460055195690522 44.39208221435547
2.108337289402277 1.9120465517044067
2.1392409410008737 2.93174409866333
2.6808361412378936 -1.9033546447753906
3.9480362951186487 43.79035186767578
